# D2.7 · Stop authority

**Function D — AI for SecOps → The Incident Responder**  ·  *Security of AI*

Builds on **[D2.6 · Post-incident change surface](https://spbreed.github.io/cyber-commons/lessons/D2.6.html)**.

| | |
|---|---|
| Open-source tooling | kagent |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

At three in the morning, the question is not what went wrong. It is who is allowed to stop it, on what evidence, without waiting for a forty-person bridge call to reach consensus.

## 2 · The framework

```
   03:00, the agent is acting, the evidence is partial

   who may say stop?          +-----------------------------+
                              | named role, on call         |
   on what evidence?          | pre-agreed trigger list     |
   what does stop mean?       | revoke + gateway cut        |
   who is told after?         | named, not assembled at 3am |
                              +-----------------------------+

   pre-agreed authority beats a forty-person bridge call
```

Stop authority is the control everyone assumes exists and almost nobody has
timed.

Five questions decide whether you have it, and each needs a name or a number
rather than an intention:

1. **Who** can halt an agent fleet without seeking approval?
2. **What** is the mechanism — and is it revocation, which survives a restart,
   or process termination, which does not?
3. **How long** does it take, measured end to end, not estimated?
4. **What breaks** when it fires — and has the business already agreed to that?
5. **Who turns it back on**, and against what evidence?

An untested stop button is a belief. The purpose of this lesson is to convert it
into a measurement, because the measurement is what an auditor, a regulator and
a board will each ask for in different words.

## 3 · Demo — the five questions, answered badly and well

In [ ]:
VAGUE = {
 "who":       "the security team",
 "mechanism": "we can turn off the agents",
 "time":      "quickly",
 "breaks":    "not much",
 "restart":   "when it's safe",
}
CONCRETE = {
 "who":       "on-call SRE, no approval required for non-human identities",
 "mechanism": "revoke the SPIFFE identity at the gateway (survives restart)",
 "time":      "measured 12s decision→first failed call, game day 2026-07-04",
 "breaks":    "auto-remediation pauses; ticket queue grows ~40/hour; "
              "agreed with the service owner 2026-05-11",
 "restart":   "security lead, after the C1.2 containment suite passes on the new build",
}
for k in VAGUE:
    print(f"{k:11s} VAGUE    {VAGUE[k]}")
    print(f"{'':11s} CONCRETE {CONCRETE[k]}\n")

## 4 · Where it breaks — mechanism matters more than speed

In [ ]:
from dataclasses import dataclass

@dataclass
class Agent:
    name: str; running: bool = True; identity_valid: bool = True
    def can_act(self): return self.running and self.identity_valid

MECHANISMS = {
 "kill the process":      (2,   lambda a: setattr(a, "running", False)),
 "network quarantine":    (5,   lambda a: None),
 "revoke the identity":   (12,  lambda a: setattr(a, "identity_valid", False)),
 "rotate the credential": (420, lambda a: setattr(a, "identity_valid", False)),
}
print(f"{'mechanism':24s}{'secs':>6}{'stops it':>10}{'survives restart':>19}")
print("-" * 60)
for name, (secs, apply) in MECHANISMS.items():
    a = Agent("patch-agent")
    apply(a)
    stopped = not a.can_act()
    a.running = True                      # a supervisor restarts the process
    survives = not a.can_act()
    print(f"{name:24s}{secs:>6}{str(stopped):>10}{str(survives):>19}")
print("\nThe fastest mechanism is the one that does not survive a restart.")
print("Speed without persistence is a pause, not a stop.")

## 5 · The control — run the game day and record the number

In [ ]:
GAME_DAY = [
 ("decision made",                    0),
 ("on-call authenticates to the IdP", 4),
 ("identity revoked",                 9),
 ("gateway cache expires",            12),
 ("agent's next call fails",          12),
 ("confirmed in telemetry",           38),
]
print(f"{'step':38s}{'t+s':>6}")
print("-" * 46)
for step, t in GAME_DAY: print(f"{step:38s}{t:>6}")
mttstop = GAME_DAY[4][1]
print(f"\nmeasured time-to-stop: {mttstop}s")
print(f"time-to-confirm:       {GAME_DAY[-1][1]}s")

def cost_of_stop(rate_per_min, seconds):
    return round(rate_per_min * seconds / 60)
for rate in (60, 300, 1200):
    print(f"   at {rate:>5}/min a {mttstop}s stop still permits "
          f"{cost_of_stop(rate, mttstop):>4} further actions")

def stop_authority_ready(answers, measured_seconds, tested_days_ago):
    problems = []
    if any(len(v.split()) < 4 for v in answers.values()):
        problems.append("at least one answer is not specific")
    if measured_seconds is None:
        problems.append("time-to-stop has never been measured")
    if tested_days_ago is None or tested_days_ago > 180:
        problems.append("not tested in the last 180 days")
    return (not problems), problems

for label, ans, secs, days in (("as usually documented", VAGUE, None, None),
                               ("after a game day", CONCRETE, 12, 41)):
    ok, problems = stop_authority_ready(ans, secs, days)
    print(f"\n{label}: ready={ok}")
    for p in problems: print(f"   ⚠ {p}")
assert stop_authority_ready(CONCRETE, 12, 41)[0]

## What you just proved

The vague and concrete answers print side by side. Killing the process stops the agent but does not survive a restart, while identity revocation does. The game-day timeline gives a measured 12-second time-to-stop, permitting 12 to 240 further actions depending on rate. The readiness check fails the vague version on three counts and passes the tested one.

## Your turn

Run the game day. The deliverable is the number, and the number is what goes in the evidence pack for E1.7 and the board slide for E3.5. An untested stop button is a belief.

---

**Next → [D2.8 · Regulatory clock](https://spbreed.github.io/cyber-commons/lessons/D2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*